# Meme Coin Data Pipeline

**All-in-one pipeline:** Fetches, updates, fills, and crops meme coin OHLCV data in the **raw data folder** (`raw_data/`).

- **Fill:** every zero-volume minute, whether filled by the API (empty intervals) or added here for a minute the API skipped, is a **flat candle at the last close**: open = high = low = close, volume 0. The `backfilled` flag marks these minutes and is re-derived on every run.
- **Updates:** only coins in `COIN_LIST` that are **not marked dead** are updated. Coins in `raw_data/` that aren't in `COIN_LIST` are never touched, and neither are coins whose metadata says `death_point.is_dead = true` (older archived coins are marked dead once, because the API only reaches about 6 months back, so updating them would leave a gap).

- Resumes from where it left off on re-run
- Updates existing coins before collecting new ones
- 20-second delay between every API call
- Stores only the final cropped + cleaned CSV and metadata

In [1]:
#
# ============================================================
# CONFIGURATION — Edit the coin list and output folder here
# ============================================================
#

# Format: (token_address, pool_address)

COIN_LIST = [
    ("..."),
]

OUTPUT_DIR = "raw_data"   # the raw data folder: new coins are written here and existing ones
                          # updated in place (coins marked dead in their metadata are skipped)
NETWORK = "solana"
TIMEZONE = "Europe/Amsterdam"
API_DELAY = 15          # seconds between every API call
BASE_URL = "https://api.geckoterminal.com/api/v2"

In [2]:
import os
import gc
import json
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime

pd.options.display.float_format = '{:.9f}'.format

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def api_sleep():
    """Universal delay after every API call."""
    time.sleep(API_DELAY)


def get_token_symbol(token_address, pool_address):
    """
    Fetch the token symbol from GeckoTerminal.
    Returns the naming key: {symbol_lower}_{pool_prefix5}
    """
    try:
        response = requests.get(
            f"{BASE_URL}/networks/{NETWORK}/tokens/{token_address}/info",
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()
        token_name = data["data"]["attributes"]["name"]
        token_symbol = data["data"]["attributes"]["symbol"]
        print(f"  Token: {token_name} ({token_symbol})")
    except Exception as e:
        print(f"  Error fetching token info: {e}")
        token_symbol = "unknown"
        token_name = "Unknown Token"
    finally:
        api_sleep()

    pool_prefix = pool_address[:5]
    return f"{token_symbol.lower()}_{pool_prefix}", token_name


def build_metadata(base_name, token_name, token_address, pool_address,
                   is_dead=False, death_timestamp=None):
    """Create the metadata dict. `is_dead`/`death_timestamp` record whether a
    48h-window death point was detected (a dead coin is skipped on future
    updates)."""
    return {
        "project_info": {
            "created_at": datetime.now().isoformat(),
            "description": f"{base_name.upper()}/SOL trading data analysis",
            "timezone": TIMEZONE,
        },
        "api_config": {
            "base_url": BASE_URL,
            "network": NETWORK,
        },
        "token_info": {
            "token_address": token_address,
            "token_symbol": base_name.split("_")[0].upper(),
            "token_name": token_name,
        },
        "pool_info": {
            "pool_address": pool_address,
            "pair": f"{base_name.split('_')[0].upper()}/SOL",
        },
        "data_files": {
            "cleaned_data": f"{OUTPUT_DIR}/{base_name}_ohlcv_cleaned.csv",
        },
        "death_point": {
            "is_dead": bool(is_dead),
            "death_timestamp": death_timestamp,
            "window_hours": 48,
            "volume_threshold_sol": 1000,
        },
        "notes": [
            "Every zero-volume minute (API empty interval or a minute the API skipped) is a flat candle at the last close (open = high = low = close), volume 0",
            "Backfilled column: 1 = zero-volume (filled) minute, 0 = a real candle; re-derived on every run",
            "Data is cropped at the death point: end of the first 48h moving window with < 1000 SOL volume (the first 48h are ignored for detection)",
            "death_point.is_dead = true means the coin is dead and is skipped on future updates",
        ],
    }


# ============================================================
# EXISTING-COIN DETECTION
# ============================================================

def scan_existing_coins():
    """
    Scan OUTPUT_DIR and return one record per coin that already has DATA on disk
    (a *_ohlcv_cleaned.csv or *_ohlcv.csv). Resume keys off the CSV — not the metadata — so a
    coin whose metadata is missing or corrupt (e.g. an interrupted save) is still
    recognised and appended-to instead of being re-fetched from scratch.

    Each record: {base_name, csv_path, meta_path, pool_address|None,
                  pool_prefix, is_dead}.
    """
    records = []
    if not os.path.exists(OUTPUT_DIR):
        return records

    seen = set()
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        base_name = None
        for suffix in ("_ohlcv_cleaned.csv", "_ohlcv.csv"):     # archived coins may use either name
            if fname.endswith(suffix):
                base_name = fname[:-len(suffix)]
                break
        if base_name is None or base_name in seen:
            continue
        seen.add(base_name)
        csv_path = os.path.join(OUTPUT_DIR, f"{base_name}_ohlcv_cleaned.csv")
        if not os.path.exists(csv_path):
            csv_path = os.path.join(OUTPUT_DIR, f"{base_name}_ohlcv.csv")
        meta_path = os.path.join(OUTPUT_DIR, f"{base_name}_metadata.json")

        pool_address, is_dead = None, False
        if os.path.exists(meta_path):
            try:
                with open(meta_path, "r") as f:
                    meta = json.load(f)
                pool_address = meta.get("pool_info", {}).get("pool_address")
                is_dead = meta.get("death_point", {}).get("is_dead", False)
            except Exception:
                pass  # corrupt metadata -> fall back to the filename pool-prefix

        records.append({
            "base_name": base_name,
            "csv_path": csv_path,
            "meta_path": meta_path,
            "pool_address": pool_address,
            "pool_prefix": base_name.rsplit("_", 1)[-1],  # the {pool_address[:5]} suffix
            "is_dead": is_dead,
        })
    return records


def match_existing(pool_address, records):
    """
    Find the on-disk coin for a COIN_LIST pool_address. Prefer an exact match on
    the pool_address stored in (readable) metadata; otherwise fall back to the
    5-char pool prefix embedded in the filename, so coins with missing/corrupt
    metadata are still recognised. Returns the record, or None if there is no
    match (or an ambiguous prefix match -> treated as new, which is the safe
    default).
    """
    for r in records:
        if r["pool_address"] == pool_address:
            return r
    prefix_hits = [r for r in records if r["pool_prefix"] == pool_address[:5]]
    if len(prefix_hits) == 1:
        return prefix_hits[0]
    return None


# ============================================================
# DATA FETCHING
# ============================================================

def _fetch_ohlcv_batch(pool_address, before_timestamp=None, limit=1000):
    """
    Single API call: fetch up to `limit` minute candles going backwards.
    Returns (ohlcv_list, error_string_or_None).
    """
    params = {
        "aggregate": "1",
        "limit": int(limit),
        "currency": "token",
        "token": "base",
        "include_empty_intervals": "true",
    }
    if before_timestamp is not None:
        params["before_timestamp"] = before_timestamp

    try:
        response = requests.get(
            f"{BASE_URL}/networks/{NETWORK}/pools/{pool_address}/ohlcv/minute",
            params=params,
            timeout=15,
        )
        response.raise_for_status()
        ohlcv_list = response.json()["data"]["attributes"]["ohlcv_list"]
        return ohlcv_list, None
    except Exception as e:
        return [], str(e)
    finally:
        api_sleep()


def _raw_to_df(all_data):
    """Convert raw OHLCV list to a sorted DataFrame."""
    if not all_data:
        return None
    df = pd.DataFrame(all_data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["timestamp"] = df["timestamp"].dt.tz_convert(TIMEZONE)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)
    df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    return df


def _get_pool_created_at(pool_address):
    """Return the pool's creation time (tz-aware, the coin's launch) from
    GeckoTerminal, or None if it can't be determined."""
    try:
        response = requests.get(
            f"{BASE_URL}/networks/{NETWORK}/pools/{pool_address}",
            timeout=10,
        )
        response.raise_for_status()
        created = response.json()["data"]["attributes"].get("pool_created_at")
        return pd.to_datetime(created, utc=True) if created else None
    except Exception as e:
        print(f"    Could not read pool_created_at: {e}")
        return None
    finally:
        api_sleep()


def _fetch_full_history_backward(pool_address):
    """Fallback: page backward (newest -> oldest) all the way to the beginning."""
    all_data = []
    before_ts = None
    low_count_streak = 0
    batch_num = 0

    while True:
        batch_num += 1
        ohlcv_list, error = _fetch_ohlcv_batch(pool_address, before_timestamp=before_ts)

        if error:
            print(f"    Batch {batch_num}: API error — {error}")
            break
        if not ohlcv_list:
            print(f"    Batch {batch_num}: No more data (reached beginning)")
            break

        n = len(ohlcv_list)
        all_data.extend(ohlcv_list)
        before_ts = ohlcv_list[-1][0]
        print(f"    Batch {batch_num}: {n} candles (total: {len(all_data)})")

        if n <= 5:
            low_count_streak += 1
            if low_count_streak >= 3:
                print(f"    Stopping: 3 consecutive batches with ≤5 candles")
                break
        else:
            low_count_streak = 0

    print(f"  Total candles fetched: {len(all_data)}")
    return _raw_to_df(all_data)


def _fetch_full_history_forward(pool_address, launch_ts,
                                window_hours=48, volume_threshold=1000):
    """
    Page FORWARD from the coin's launch, running the death check after every
    batch. Because the accumulated data starts at the true launch, the 48h
    grace period is correctly anchored, so the FIRST 48h-window death point we
    find is the real one. As soon as it appears we stop — the flat dead tail
    after it would only be cropped off later, so we never waste API calls
    downloading it. (A per-batch check on a BACKWARD fetch can't do this: it
    would trip on the recent tail and discard the coin's early history.)
    """
    BATCH_SEC = 1000 * 60                      # 1000 one-minute candles per call
    launch_sec = int(launch_ts.timestamp())
    now_sec = int(pd.Timestamp.now(tz="UTC").timestamp())

    all_data = []
    cursor = launch_sec + BATCH_SEC            # before_timestamp of the first forward page
    batch_num = 0
    last_newest = -1                           # guards against a stalled (non-advancing) page

    while cursor <= now_sec + BATCH_SEC:
        batch_num += 1
        ohlcv_list, error = _fetch_ohlcv_batch(pool_address, before_timestamp=cursor)

        if error:
            print(f"    Batch {batch_num}: API error — {error}")
            break

        if ohlcv_list:
            all_data.extend(ohlcv_list)
            newest = max(row[0] for row in ohlcv_list)
            print(f"    Batch {batch_num}: {len(ohlcv_list)} candles (total: {len(all_data)})")
        else:
            newest = cursor
            print(f"    Batch {batch_num}: no candles in window")

        # Death check anchored to the true launch. include_empty_intervals keeps
        # the series contiguous, so find_death_point can run on the raw frame.
        df = _raw_to_df(all_data)
        if df is not None and len(df) >= window_hours * 60:
            death_ts, _ = find_death_point(df, window_hours=window_hours,
                                           volume_threshold=volume_threshold)
            if death_ts is not None:
                print(f"    Death point reached at {death_ts} — stopping (dead tail skipped)")
                break

        if newest >= now_sec:
            print(f"    Reached present — full history fetched")
            break
        if newest <= last_newest:              # no forward progress -> stop (avoid stalling)
            print(f"    Fetch stalled (no new candles) — stopping")
            break
        last_newest = newest
        cursor = newest + BATCH_SEC

    print(f"  Total candles fetched: {len(all_data)}")
    return _raw_to_df(all_data)


def fetch_full_history(pool_address):
    """
    Fetch a coin's history. Prefers forward paging from the pool's launch with
    per-batch death detection, so a long flat dead tail is never downloaded.
    Falls back to a full backward fetch if the launch time is unavailable.
    """
    launch_ts = _get_pool_created_at(pool_address)
    if launch_ts is not None:
        print(f"  Launch (pool_created_at): {launch_ts} — fetching forward with death check")
        return _fetch_full_history_forward(pool_address, launch_ts)
    print(f"  Launch time unknown — falling back to full backward fetch")
    return _fetch_full_history_backward(pool_address)


def fetch_recent_since(pool_address, cutoff, buffer_minutes=10, max_batches=60):
    """
    Fetch only the candles newer than `cutoff` (plus a small buffer), paging
    backward from now and STOPPING as soon as we have reached back past cutoff.
    A small gap costs a single small call instead of a fixed 2000-candle fetch,
    so frequent updates don't re-download data you already have.
    """
    all_data = []
    before_ts = None

    for i in range(max_batches):
        # How far back do we still need to reach (from now, or from the oldest
        # candle fetched so far) to cover `cutoff`?
        if before_ts is None:
            anchor = pd.Timestamp.now(tz=TIMEZONE)
        else:
            anchor = pd.to_datetime(before_ts, unit="s", utc=True).tz_convert(TIMEZONE)
        gap_minutes = (anchor - cutoff).total_seconds() / 60.0
        if gap_minutes <= 0:
            break

        limit = int(min(1000, gap_minutes + buffer_minutes))
        limit = max(limit, 1)

        ohlcv_list, error = _fetch_ohlcv_batch(pool_address, before_timestamp=before_ts, limit=limit)
        if error:
            print(f"    Update fetch {i+1}: API error — {error}")
            break
        if not ohlcv_list:
            print(f"    Update fetch {i+1}: No data returned")
            break

        all_data.extend(ohlcv_list)
        before_ts = ohlcv_list[-1][0]
        oldest_ts = pd.to_datetime(before_ts, unit="s", utc=True).tz_convert(TIMEZONE)
        print(f"    Update fetch {i+1}: {len(ohlcv_list)} candles (back to {oldest_ts})")

        if oldest_ts <= cutoff:                # reached back past the last saved candle
            break

    return _raw_to_df(all_data)


# ============================================================
# UPDATE EXISTING COIN
# ============================================================

def update_existing_coin(pool_address, csv_path):
    """
    Update an already-fetched coin with the latest candles.
    Drops the 5 most recent rows (may be incomplete), then fetches only the
    candles newer than the last saved one and merges.
    """
    df = pd.read_csv(csv_path)
    # If the cleaned CSV has a 'backfilled' column, drop it for merging
    if "backfilled" in df.columns:
        df = df.drop(columns=["backfilled"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["timestamp"] = df["timestamp"].dt.tz_convert(TIMEZONE)

    print(f"  Existing data: {len(df)} rows")
    print(f"  Latest timestamp: {df['timestamp'].max()}")

    # Drop 5 most recent rows
    df = df.sort_values("timestamp").iloc[:-5].reset_index(drop=True)
    cutoff = df["timestamp"].max()
    print(f"  After dropping 5 newest: {len(df)} rows (cutoff {cutoff})")

    # Fetch only what's newer than the cutoff
    recent_df = fetch_recent_since(pool_address, cutoff)
    if recent_df is None or len(recent_df) == 0:
        print("  No recent data fetched")
        return df

    recent_df = recent_df[recent_df["timestamp"] > cutoff]
    if len(recent_df) == 0:
        print("  Already up to date")
        return df

    combined = pd.concat([df, recent_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    print(f"  Updated to {len(combined)} rows (latest: {combined['timestamp'].max()})")
    return combined


# ============================================================
# FORWARD-FILL (BACKFILL MISSING MINUTES)
# ============================================================

def backfill_missing_candles(df):
    """
    Put the series on a continuous 1-minute grid and make EVERY zero-volume minute a
    FLAT candle at the last close: open = high = low = close = the last real close,
    volume 0 (the same as the live bot). That covers minutes the API skipped (added
    here) and the API's own empty intervals. Adds 'backfilled' = 1 for zero-volume
    minutes, 0 for real candles; it is derived from the volume, so it stays correct
    however often a coin is updated.
    """
    if df is None or len(df) == 0:
        return df

    df = df.drop(columns=["backfilled"], errors="ignore")
    df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    full_range = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="1min",
    )

    missing_count = len(full_range) - len(df)
    print(f"  Missing minutes to fill: {missing_count}")

    df_filled = df.set_index("timestamp").reindex(full_range)
    df_filled["volume"] = df_filled["volume"].fillna(0)
    zero = (df_filled["volume"] <= 0).to_numpy()
    last_close = df_filled["close"].where(~zero).ffill()          # last REAL close
    last_close = last_close.fillna(df_filled["close"].bfill())    # zero-volume minutes before any real candle
    for col in ("open", "high", "low", "close"):
        df_filled.loc[zero, col] = last_close[zero]
    df_filled["backfilled"] = zero.astype(int)
    df_filled = df_filled.reset_index().rename(columns={"index": "timestamp"})

    print(f"  After fill: {len(df_filled)} rows ({int(zero.sum())} zero-volume minutes, flat at the last close)")
    return df_filled


# ============================================================
# CROP (DEATH-POINT DETECTION)
# ============================================================

def find_death_point(df, window_hours=48, volume_threshold=1000, ignore_hours=48):
    """
    Slide a `window_hours` window minute-by-minute over the data and return the
    FIRST window whose total volume is below `volume_threshold` (in Solana).
    That window's END timestamp is the death point: all data from it forward is
    dropped and the coin is marked dead.

    The first `ignore_hours` of data are IGNORED for death detection (a launch
    grace period), so a slow start can't flag the coin dead — only windows that
    BEGIN at or after the `ignore_hours` mark are considered.

    Assumes a continuous 1-minute grid, so call this AFTER
    backfill_missing_candles. Returns (death_ts, window_vol) or (None, None).
    """
    if df is None or len(df) == 0:
        return None, None

    df = df.sort_values("timestamp").reset_index(drop=True)
    window_minutes = int(window_hours * 60)
    ignore_minutes = int(ignore_hours * 60)
    if len(df) < window_minutes:
        return None, None

    vol = df["volume"].to_numpy(dtype=float)
    csum = np.concatenate(([0.0], np.cumsum(vol)))
    # win_sums[k] = volume over rows [k, k+window_minutes-1]; k is the window's
    # START row, so requiring k >= ignore_minutes ignores the first 48h of data.
    win_sums = csum[window_minutes:] - csum[:-window_minutes]

    below = np.where(win_sums < volume_threshold)[0]
    below = below[below >= ignore_minutes]   # skip windows starting in the grace period
    if len(below) == 0:
        return None, None

    k = int(below[0])
    end_i = k + window_minutes - 1            # last row of the dead window
    return df.loc[end_i, "timestamp"], float(win_sums[k])


def crop_at_death_point(df):
    """
    Crop the DataFrame at the death point (end of the first 48h window with
    < 1000 Solana volume). Returns (df_cropped, was_cropped, death_ts).
    was_cropped is True exactly when a death point was found (i.e. the coin is
    now dead).
    """
    death_ts, window_vol = find_death_point(df)

    if death_ts is not None:
        before = len(df)
        df_cropped = df[df["timestamp"] <= death_ts].copy()
        print(f"  Death point at {death_ts} (48h window vol {window_vol:.2f} < 1000 SOL)")
        print(f"  Cropped {before} -> {len(df_cropped)} rows")
        return df_cropped, True, death_ts
    else:
        print(f"  No death point detected — keeping all {len(df)} rows")
        return df, False, None

# ============================================================
# SAVE FINAL OUTPUT
# ============================================================

def _atomic_write_csv(df, path):
    """Write a CSV atomically: full write to a temp file, then rename into place,
    so an interruption can never leave a half-written / corrupt file behind."""
    tmp = f"{path}.tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def _atomic_write_json(obj, path):
    """Write JSON atomically (temp file + rename)."""
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def save_coin(df, base_name, token_name, token_address, pool_address,
              is_dead=False, death_timestamp=None):
    """Save cropped CSV and metadata to OUTPUT_DIR. Both writes are atomic, so an
    interrupt can't corrupt either file and break resume on the next run."""
    csv_path = os.path.join(OUTPUT_DIR, f"{base_name}_ohlcv_cleaned.csv")
    meta_path = os.path.join(OUTPUT_DIR, f"{base_name}_metadata.json")

    _atomic_write_csv(df, csv_path)
    print(f"  Saved CSV  -> {csv_path}")

    metadata = build_metadata(base_name, token_name, token_address, pool_address,
                              is_dead=is_dead, death_timestamp=death_timestamp)
    _atomic_write_json(metadata, meta_path)
    print(f"  Saved meta -> {meta_path}")

# ============================================================
# MAIN PIPELINE
# ============================================================

def run_pipeline():
    print("=" * 80)
    print("MEME COIN DATA PIPELINE")
    print("=" * 80)
    print(f"Output directory : {OUTPUT_DIR}")
    print(f"Coins in list   : {len(COIN_LIST)}")
    print(f"API delay       : {API_DELAY}s between every call")
    print()


    # Drop duplicate (token_address, pool_address) pairs, keeping only the first
    # occurrence. Duplicates are printed (by the first 5 chars of the token
    # address) so they can be removed from COIN_LIST manually.
    seen = set()
    coin_list = []
    duplicates = []
    for token_addr, pool_addr in COIN_LIST:
        key = (token_addr, pool_addr)
        if key in seen:
            duplicates.append((token_addr, pool_addr))
        else:
            seen.add(key)
            coin_list.append((token_addr, pool_addr))

    if duplicates:
        print(f"Duplicate (token+pool) pairs skipped: {len(duplicates)} "
              f"(first occurrence of each is kept)")
        for token_addr, pool_addr in duplicates:
            print(f"  token {token_addr[:5]}...  pool {pool_addr[:5]}...  — duplicate, skipped")
        print()

    # Discover what data is already on disk (keyed by the CSV, so it is robust
    # to metadata that is missing or corrupt from an interrupted save).
    existing_records = scan_existing_coins()
    print(f"Already on disk : {len(existing_records)} coin(s) with data")
    for r in existing_records:
        tag = " [DEAD]" if r["is_dead"] else ""
        if r["pool_address"]:
            meta_tag = ""
        elif r["is_dead"]:
            meta_tag = " (no pool address in metadata — dead, so never updated)"
        else:
            meta_tag = " (metadata missing/corrupt — will be rebuilt if the coin is in COIN_LIST)"
        print(f"  {r['base_name']}{tag}{meta_tag}")
    print()

    # Split COIN_LIST into: skip (already dead), update (append), fetch (new).
    to_update = []
    to_fetch  = []
    dead_skipped = []
    data_skipped = []
    for token_addr, pool_addr in coin_list:
        rec = match_existing(pool_addr, existing_records)
        if rec is None:
            to_fetch.append((token_addr, pool_addr))
        elif rec["is_dead"]:
            dead_skipped.append((token_addr, pool_addr, rec["base_name"]))
        else:
            to_update.append((token_addr, pool_addr, rec["base_name"],
                              rec["csv_path"], rec["meta_path"]))

    print(f"To update     : {len(to_update)}")
    print(f"To fetch (new): {len(to_fetch)}")
    print(f"Dead (skipped): {len(dead_skipped)}")
    for _, _, base_name in dead_skipped:
        print(f"  {base_name} — marked dead, not updated")
    print("=" * 80)

    # ----------------------------------------------------------
    # PHASE 1 — Update existing coins
    # ----------------------------------------------------------
    if to_update:
        print("\n" + "=" * 80)
        print("PHASE 1: UPDATING EXISTING COINS")
        print("=" * 80)

    for token_addr, pool_addr, base_name, csv_path, meta_path in to_update:
        print(f"\n--- {base_name} ---")

        # token_name is cosmetic; read it from metadata if available. A coin
        # recovered via CSV-only (corrupt metadata) will get valid metadata
        # rewritten by save_coin below.
        token_name = "Unknown Token"
        if os.path.exists(meta_path):
            try:
                with open(meta_path, "r") as f:
                    token_name = json.load(f).get("token_info", {}).get("token_name", "Unknown Token")
            except Exception:
                pass

        try:
            # 1. Update raw data (append recent candles to the existing CSV)
            df = update_existing_coin(pool_addr, csv_path)

            # 2. Forward-fill
            df = backfill_missing_candles(df)

            # 3. Crop (detect 48h-window death point)
            df, was_cropped, death_ts = crop_at_death_point(df)

            # 4. Save (mark dead if a death point was found)
            save_coin(df, base_name, token_name, token_addr, pool_addr,
                      is_dead=was_cropped,
                      death_timestamp=death_ts.isoformat() if death_ts is not None else None)

        except Exception as e:
            print(f"  ERROR updating {base_name}: {e}")
        finally:
            # Free memory
            try:
                del df
            except NameError:
                pass
            gc.collect()
            print(f"  Memory freed.")

    # ----------------------------------------------------------
    # PHASE 2 — Fetch new coins
    # ----------------------------------------------------------
    if to_fetch:
        print("\n" + "=" * 80)
        print("PHASE 2: FETCHING NEW COINS")
        print("=" * 80)

    for token_addr, pool_addr in to_fetch:
        print(f"\n--- New coin: pool {pool_addr[:12]}... ---")

        try:
            # 1. Resolve token symbol
            base_name, token_name = get_token_symbol(token_addr, pool_addr)
            print(f"  File key: {base_name}")

            # 2. Fetch full history
            print(f"  Fetching full history...")
            df = fetch_full_history(pool_addr)

            if df is None or len(df) == 0:
                print(f"  SKIPPED — no data returned")
                continue

            print(f"  Raw data: {len(df)} rows")
            print(f"  Range: {df['timestamp'].min()} to {df['timestamp'].max()}")

            # 3. Forward-fill
            df = backfill_missing_candles(df)

            # 4. Crop (detect 48h-window death point)
            df, was_cropped, death_ts = crop_at_death_point(df)

            # 5. Save (mark dead if a death point was found)
            save_coin(df, base_name, token_name, token_addr, pool_addr,
                      is_dead=was_cropped,
                      death_timestamp=death_ts.isoformat() if death_ts is not None else None)

        except Exception as e:
            print(f"  ERROR fetching coin: {e}")
        finally:
            # Free memory
            try:
                del df
            except NameError:
                pass
            gc.collect()
            print(f"  Memory freed.")

    # ----------------------------------------------------------
    # DONE
    # ----------------------------------------------------------
    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE")
    print("=" * 80)
    final_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith("_ohlcv_cleaned.csv")]
    dead_count = 0
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith("_metadata.json"):
            try:
                with open(os.path.join(OUTPUT_DIR, f)) as fh:
                    if json.load(fh).get("death_point", {}).get("is_dead", False):
                        dead_count += 1
            except Exception:
                pass
    print(f"Coins in {OUTPUT_DIR}/: {len(final_files)}  (marked dead: {dead_count})")
    for f in sorted(final_files):
        print(f"  {f}")


run_pipeline()

MEME COIN DATA PIPELINE
Output directory : raw_data
Coins in list   : 257
API delay       : 15s between every call

Duplicate (token+pool) pairs skipped: 1 (first occurrence of each is kept)
  token 2j5Sa...  pool Fe7bp...  — duplicate, skipped

Already on disk : 407 coin(s) with data
  $hachi_DqAfr [DEAD]
  114514_aa2x8 [DEAD] (no pool address in metadata — dead, so never updated)
  2026_4GfcD [DEAD]
  2026moon_GA7V9 [DEAD]
  3.1459_7djya [DEAD] (no pool address in metadata — dead, so never updated)
  3place_Fyuo7 [DEAD]
  51_9E7DK [DEAD]
  67_DMAFL [DEAD]
  6ixrooms_3DsQy [DEAD]
  ace_6FEUg [DEAD]
  agamemnon_636bk [DEAD]
  agi_ETWmZ [DEAD]
  agi_Fej6x [DEAD]
  ai16z_7qAVr [DEAD]
  ainl_6DNmW [DEAD]
  albatross_7ipTH [DEAD]
  all_uzAK8 [DEAD]
  allinu_4iVHk [DEAD]
  ameen_9ezrx [DEAD] (no pool address in metadata — dead, so never updated)
  anonymouse_CBfWf [DEAD]
  ansem_6e7V9
  ansem_FnzKY
  app_95xZP [DEAD]
  apple_4AKhv [DEAD]
  apu_2jtXJ [DEAD]
  arc_J3b6d [DEAD]
  archibrown_4g

In [3]:
# ============================================================
# DATA QUALITY CHECK — All coins in pipeline_output/
# ============================================================

import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "raw_data"
QUALITY_THRESHOLD_EXTREME_PCT = 90  # % price move in one minute

csv_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith("_ohlcv_cleaned.csv")])
print(f"Found {len(csv_files)} cleaned CSV file(s) in {OUTPUT_DIR}/\n")

summary_rows = []

for csv_file in csv_files:
    path = os.path.join(OUTPUT_DIR, csv_file)
    base_name = csv_file.replace("_ohlcv_cleaned.csv", "")

    print("=" * 80)
    print(f"  {base_name}")
    print("=" * 80)

    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)

    first_ts = df["timestamp"].min()
    last_ts = df["timestamp"].max()
    numeric_cols = df.columns.drop(["timestamp", "backfilled"] if "backfilled" in df.columns else ["timestamp"])

    # 1. DUPLICATES
    dup_count = df.duplicated(subset=["timestamp"], keep=False).sum()
    print(f"\n1. Duplicate timestamps: {dup_count}")
    if dup_count > 0:
        dups = df[df.duplicated(subset=["timestamp"], keep=False)]
        for ts in dups["timestamp"].unique()[:5]:
            print(f"   - {ts}")
        if dups["timestamp"].nunique() > 5:
            print(f"   ... and {dups['timestamp'].nunique() - 5} more")

    # 2. MISSING TIMESTAMPS
    expected = pd.date_range(start=first_ts, end=last_ts, freq="1min", tz="UTC")
    missing_count = len(expected) - len(df)
    print(f"\n2. Missing timestamps: {missing_count}  (expected {len(expected)}, got {len(df)})")

    # 3. MISSING / INVALID VALUES
    null_total = df[numeric_cols].isnull().sum().sum()
    neg_total = (df[numeric_cols] < 0).sum().sum()
    inf_total = np.isinf(df[numeric_cols]).sum().sum()
    zero_vol = (df["volume"] == 0).sum() if "volume" in df.columns else 0
    print(f"\n3. Null values: {null_total}  |  Negatives: {neg_total}  |  Infs: {inf_total}")
    print(f"   Zero-volume rows: {zero_vol} ({zero_vol / len(df) * 100:.1f}%)")

    # 4. OHLC CONSISTENCY
    ohlc_issues = 0
    if all(c in df.columns for c in ["open", "high", "low", "close"]):
        ohlc_issues = (
            (df["high"] < df["low"]).sum()
            + (df["high"] < df["open"]).sum()
            + (df["high"] < df["close"]).sum()
            + (df["low"] > df["open"]).sum()
            + (df["low"] > df["close"]).sum()
        )
    print(f"\n4. OHLC consistency violations: {ohlc_issues}")

    # 5. EXTREME PRICE MOVEMENTS
    extreme_moves = 0
    if "close" in df.columns:
        pct = df["close"].pct_change().abs() * 100
        extreme_moves = (pct > QUALITY_THRESHOLD_EXTREME_PCT).sum()
        if extreme_moves > 0:
            print(f"\n5. Extreme moves (>{QUALITY_THRESHOLD_EXTREME_PCT}%): {extreme_moves}")
            for idx in df.index[pct > QUALITY_THRESHOLD_EXTREME_PCT][:5]:
                print(f"   Row {idx} ({df.loc[idx, 'timestamp']}): {pct[idx]:.1f}%")
            if extreme_moves > 5:
                print(f"   ... and {extreme_moves - 5} more")
        else:
            print(f"\n5. Extreme moves (>{QUALITY_THRESHOLD_EXTREME_PCT}%): 0")

    # 6. VOLUME ANALYSIS
    if "volume" in df.columns:
        print(f"\n6. Volume — total: {df['volume'].sum():,.2f}  |  mean: {df['volume'].mean():,.2f}  "
              f"|  median: {df['volume'].median():,.2f}  |  max: {df['volume'].max():,.2f}")

    # 7. QUALITY SCORE
    quality = 100.0
    if len(expected) > 0:
        quality -= min(missing_count / len(expected) * 30, 30)
    if len(df) > 0:
        quality -= min(ohlc_issues / len(df) * 40, 40)
        quality -= min(extreme_moves / len(df) * 30, 30)

    label = ("EXCELLENT" if quality >= 90 else
             "GOOD" if quality >= 75 else
             "FAIR" if quality >= 60 else
             "POOR")
    print(f"\n   Quality score: {quality:.1f}/100 — {label}")

    summary_rows.append({
        "coin": base_name,
        "rows": len(df),
        "days": (last_ts - first_ts).days,
        "duplicates": dup_count,
        "missing": missing_count,
        "nulls": null_total,
        "ohlc_issues": ohlc_issues,
        "extreme_moves": extreme_moves,
        "quality": round(quality, 1),
    })
    print()

# OVERALL SUMMARY TABLE
print("=" * 80)
print("SUMMARY")
print("=" * 80)
summary_df = pd.DataFrame(summary_rows)
if len(summary_df) > 0:
    print(summary_df.to_string(index=False))
else:
    print("No files found.")

Found 418 cleaned CSV file(s) in raw_data/

  $hachi_DqAfr

1. Duplicate timestamps: 0

2. Missing timestamps: 0  (expected 60410, got 60410)

3. Null values: 0  |  Negatives: 0  |  Infs: 0
   Zero-volume rows: 22181 (36.7%)

4. OHLC consistency violations: 0

5. Extreme moves (>90%): 0

6. Volume — total: 101,626.26  |  mean: 1.68  |  median: 0.11  |  max: 243.58

   Quality score: 100.0/100 — EXCELLENT

  114514_aa2x8

1. Duplicate timestamps: 0

2. Missing timestamps: 0  (expected 5760, got 5760)

3. Null values: 0  |  Negatives: 0  |  Infs: 0
   Zero-volume rows: 4764 (82.7%)

4. OHLC consistency violations: 0

5. Extreme moves (>90%): 0

6. Volume — total: 3,165.26  |  mean: 0.55  |  median: 0.00  |  max: 249.64

   Quality score: 100.0/100 — EXCELLENT

  2026_4GfcD

1. Duplicate timestamps: 0

2. Missing timestamps: 0  (expected 36720, got 36720)

3. Null values: 0  |  Negatives: 0  |  Infs: 0
   Zero-volume rows: 10387 (28.3%)

4. OHLC consistency violations: 0

5. Extreme moves

In [4]:
import os
import json

OUTPUT_DIR = "raw_data"

pairs = []
for fname in sorted(os.listdir(OUTPUT_DIR)):
    if fname.endswith("_metadata.json"):
        try:
            with open(os.path.join(OUTPUT_DIR, fname), "r") as f:
                meta = json.load(f)
            token_addr = meta.get("token_info", {}).get("token_address", "")
            pool_addr = meta.get("pool_info", {}).get("pool_address", "")
            pairs.append((token_addr, pool_addr))
        except Exception as e:
            print(f"# skipped {fname}: {e}")

print(f"# {len(pairs)} coin(s) in '{OUTPUT_DIR}'")
for token_addr, pool_addr in pairs:
    print(f'    ("{token_addr}", "{pool_addr}"),')

# 418 coin(s) in 'raw_data'
    ("x95HN3DWvbfCBtTjGm587z8suK3ec6cwQwgZNLbWKyp", "DqAfrGV2GBxpGRsq6Xk1z9ojRncqgLeeVPaKg5bCc24Z"),
    ("None", "None"),
    ("CQ1eU5VLsw2WyaDgVJPDBb1RyroeZoJaMeRaNkwtpump", "4GfcD6RZK8cjqEzWcnFjsp9WvA3ZSTkSwvByQvaAJ5HP"),
    ("3bCDKrN4pyVEAvzNkH7fX5cMYo9DXwRZ5jCi3Dm8pump", "GA7V9JwjTPgVALWACf7oMFtmqS8rnhk983N6pWhgKjCJ"),
    ("None", "None"),
    ("7K4jGRV7PY1EymiY3jx75JSfi1jjzG7Gh5XHqF2Upump", "Fyuo73hXJif3W8qEagbKXUmp1RCaRhh3j81cjCrqMPv3"),
    ("2rx71CSxf8vc89jAo9zj29WY6uKpvqMNBn8cSqh6pump", "9E7DKQx89UVSi3rzPqeRuQjdwVDqN5ytvJQLQmCFkEb1"),
    ("9AvytnUKsLxPxFHFqS6VLxaxt5p6BhYNr53SD2Chpump", "DMAFL613XTipuA3jFNYczavWT7XsiYf9cR3qmRMZQhB6"),
    ("EhNEwXZq89viTNhADmm5CT8GGHxF27694mzMUVjrpump", "3DsQypPCoeiQzXQWGK32eDy5eXrNsgVpowwLAPuJw6cu"),
    ("GnHpRsrcyfHSMZNzmpjAzTFQA26vnbRMzbKQ11ZKpump", "6FEUgxZHVy2Zh4rTnoaret4Hyh1wXa3E3isouFBSSakJ"),
    ("2cAtqsRafKS7baN3mvJARhyZiMRdW4fZYNUUWUrCpump", "636bkx7Ugs6Vdb9FhAJdwFdi4afupHDarrTW2nTVuEag"),
    ("3qwtM